# STRAT-002 v5 Walk-Forward Validation with VectorBT

Comprehensive out-of-sample testing:
1. Full period baseline
2. Year-by-year performance
3. Rolling walk-forward (2yr train, 1yr test)
4. Recent performance (last 2 years)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

print(f"VectorBT version: {vbt.__version__}")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df = df.dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
def get_entries(data):
    """STRAT-002 v5 entry signals"""
    cond = (data['sopr'] < 1) & (data['sopr_sth'] < 1) & (data['rl_zscore'] > 0.5)
    return cond & ~cond.shift(1).fillna(False)

def run_vbt(data, entries, init_cash=100000):
    """Run VectorBT with 30% trailing stop"""
    if entries.sum() == 0:
        return None
    return vbt.Portfolio.from_signals(
        close=data['price'],
        entries=entries,
        exits=None,
        sl_stop=0.30,
        sl_trail=True,
        stop_exit_price='close',
        fees=0.001,
        init_cash=init_cash,
        freq='D'
    )

---
## 1. Full Period Baseline

In [ ]:
full_data = df[df.index >= '2019-01-01'].copy()
entries = get_entries(full_data)
pf = run_vbt(full_data, entries)

print("FULL PERIOD (2019-2026)")
print("="*60)
print(f"Total Return: {pf.total_return()*100:+,.0f}%")
print(f"CAGR: {pf.annualized_return()*100:+.1f}%")
print(f"Sharpe: {pf.sharpe_ratio():.2f}")
print(f"Sortino: {pf.sortino_ratio():.2f}")
print(f"Max DD: {pf.max_drawdown()*100:.1f}%")
print(f"Win Rate: {pf.trades.win_rate()*100:.0f}%")
print(f"Trades: {pf.trades.count()}")
print(f"Final: ${pf.final_value():,.0f}")

bh = (full_data['price'].iloc[-1] / full_data['price'].iloc[0] - 1) * 100
print(f"\nBuy & Hold: {bh:+,.0f}%")
print(f"Alpha: {pf.total_return()*100 - bh:+,.0f}%")

---
## 2. Year-by-Year Performance

In [ ]:
print("YEAR-BY-YEAR PERFORMANCE")
print("="*100)
print(f"{'Year':<8} {'Strategy':>12} {'Buy&Hold':>12} {'Beat?':>8} {'Trades':>8} {'Win%':>8} {'Sharpe':>8}")
print("-"*100)

years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
yearly_results = []

for year in years:
    year_data = df[(df.index >= f'{year}-01-01') & (df.index <= f'{year}-12-31')].copy()
    if len(year_data) < 30:
        continue
    
    entries = get_entries(year_data)
    pf = run_vbt(year_data, entries)
    
    bh_ret = (year_data['price'].iloc[-1] / year_data['price'].iloc[0] - 1)
    
    if pf and pf.trades.count() > 0:
        strat_ret = pf.total_return()
        trades = pf.trades.count()
        win_rate = pf.trades.win_rate()
        sharpe = pf.sharpe_ratio()
        beat = strat_ret > bh_ret
    else:
        strat_ret, trades, win_rate, sharpe, beat = 0, 0, 0, 0, False
    
    yearly_results.append({'year': year, 'strat': strat_ret, 'bh': bh_ret, 'beat': beat, 'trades': trades})
    
    beat_str = '✅' if beat else '❌'
    print(f"{year:<8} {strat_ret*100:>+11.0f}% {bh_ret*100:>+11.0f}% {beat_str:>8} {trades:>8} {win_rate*100:>7.0f}% {sharpe:>8.2f}")

print("-"*100)
beat_count = sum(1 for r in yearly_results if r['beat'])
print(f"Beat Rate: {beat_count}/{len(yearly_results)} years ({beat_count/len(yearly_results)*100:.0f}%)")

---
## 3. Rolling Walk-Forward Test

In [ ]:
print("ROLLING WALK-FORWARD (2yr train, 1yr test)")
print("="*100)
print(f"{'Test Period':<20} {'Strategy':>12} {'Buy&Hold':>12} {'Beat?':>8} {'Trades':>8} {'Entries':>8}")
print("-"*100)

wf_results = []

# Walk-forward windows
windows = [
    ('2021-01-01', '2021-12-31'),
    ('2022-01-01', '2022-12-31'),
    ('2023-01-01', '2023-12-31'),
    ('2024-01-01', '2024-12-31'),
    ('2025-01-01', '2025-12-31'),
]

for test_start, test_end in windows:
    test_data = df[(df.index >= test_start) & (df.index <= test_end)].copy()
    if len(test_data) < 30:
        continue
    
    entries = get_entries(test_data)
    pf = run_vbt(test_data, entries)
    
    bh_ret = (test_data['price'].iloc[-1] / test_data['price'].iloc[0] - 1)
    
    if pf and pf.trades.count() > 0:
        strat_ret = pf.total_return()
        trades = pf.trades.count()
        beat = strat_ret > bh_ret
    else:
        strat_ret, trades, beat = 0, 0, False
    
    wf_results.append({'period': test_start[:4], 'strat': strat_ret, 'bh': bh_ret, 'beat': beat})
    
    beat_str = '✅' if beat else '❌'
    print(f"{test_start[:4]:<20} {strat_ret*100:>+11.0f}% {bh_ret*100:>+11.0f}% {beat_str:>8} {trades:>8} {entries.sum():>8}")

print("-"*100)
wf_beat = sum(1 for r in wf_results if r['beat'])
print(f"Walk-Forward Beat Rate: {wf_beat}/{len(wf_results)} ({wf_beat/len(wf_results)*100:.0f}%)")

---
## 4. Last 2 Years (Recent Performance)

In [ ]:
recent_data = df[df.index >= '2024-01-01'].copy()
entries = get_entries(recent_data)
pf_recent = run_vbt(recent_data, entries)

print("LAST 2 YEARS (2024-2026)")
print("="*60)

if pf_recent and pf_recent.trades.count() > 0:
    print(f"Total Return: {pf_recent.total_return()*100:+,.1f}%")
    print(f"Max DD: {pf_recent.max_drawdown()*100:.1f}%")
    print(f"Trades: {pf_recent.trades.count()}")
    print(f"Win Rate: {pf_recent.trades.win_rate()*100:.0f}%")
    print(f"Final: ${pf_recent.final_value():,.0f}")
    
    bh = (recent_data['price'].iloc[-1] / recent_data['price'].iloc[0] - 1) * 100
    print(f"\nBuy & Hold: {bh:+,.1f}%")
    print(f"Strategy vs B&H: {pf_recent.total_return()*100 - bh:+,.1f}%")
else:
    print("No trades in this period")
    print(f"Entry signals: {entries.sum()}")

In [ ]:
# Trade details for recent period
if pf_recent and pf_recent.trades.count() > 0:
    print("\nRECENT TRADES")
    print("="*80)
    print(pf_recent.trades.records_readable.to_string())

---
## 5. Half-Period Test (First Half vs Second Half)

In [ ]:
# Split data in half
full_data = df[df.index >= '2019-01-01'].copy()
mid_point = full_data.index[len(full_data)//2]

first_half = full_data[full_data.index < mid_point].copy()
second_half = full_data[full_data.index >= mid_point].copy()

print("HALF-PERIOD TEST")
print("="*80)

for name, data in [('First Half', first_half), ('Second Half', second_half)]:
    entries = get_entries(data)
    pf = run_vbt(data, entries)
    
    bh = (data['price'].iloc[-1] / data['price'].iloc[0] - 1)
    
    print(f"\n{name} ({data.index.min().date()} to {data.index.max().date()})")
    print("-"*60)
    
    if pf and pf.trades.count() > 0:
        print(f"  Strategy: {pf.total_return()*100:+,.0f}%")
        print(f"  Buy&Hold: {bh*100:+,.0f}%")
        print(f"  Beat: {'✅' if pf.total_return() > bh else '❌'}")
        print(f"  Trades: {pf.trades.count()}")
        print(f"  Sharpe: {pf.sharpe_ratio():.2f}")
    else:
        print(f"  No trades (entries: {entries.sum()})")

---
## 6. Bull vs Bear Market Test

In [ ]:
# Define market periods
periods = [
    ('2019 Recovery', '2019-01-01', '2019-06-30', 'Bull'),
    ('2019 Correction', '2019-07-01', '2019-12-31', 'Bear'),
    ('2020 COVID Crash', '2020-01-01', '2020-03-31', 'Bear'),
    ('2020-21 Bull Run', '2020-04-01', '2021-04-30', 'Bull'),
    ('2021 Summer Crash', '2021-05-01', '2021-07-31', 'Bear'),
    ('2021 ATH Run', '2021-08-01', '2021-11-30', 'Bull'),
    ('2022 Bear Market', '2022-01-01', '2022-12-31', 'Bear'),
    ('2023 Recovery', '2023-01-01', '2023-12-31', 'Bull'),
    ('2024 Bull Run', '2024-01-01', '2024-12-31', 'Bull'),
]

print("MARKET REGIME PERFORMANCE")
print("="*110)
print(f"{'Period':<25} {'Type':<6} {'Strategy':>12} {'Buy&Hold':>12} {'Beat?':>8} {'Trades':>8}")
print("-"*110)

bull_results, bear_results = [], []

for name, start, end, regime in periods:
    period_data = df[(df.index >= start) & (df.index <= end)].copy()
    if len(period_data) < 10:
        continue
    
    entries = get_entries(period_data)
    pf = run_vbt(period_data, entries)
    
    bh = (period_data['price'].iloc[-1] / period_data['price'].iloc[0] - 1)
    
    if pf and pf.trades.count() > 0:
        strat = pf.total_return()
        trades = pf.trades.count()
        beat = strat > bh
    else:
        strat, trades, beat = 0, 0, bh < 0  # No trades = 0% return, beats if B&H negative
    
    result = {'name': name, 'strat': strat, 'bh': bh, 'beat': beat}
    if regime == 'Bull':
        bull_results.append(result)
    else:
        bear_results.append(result)
    
    beat_str = '✅' if beat else '❌'
    print(f"{name:<25} {regime:<6} {strat*100:>+11.0f}% {bh*100:>+11.0f}% {beat_str:>8} {trades:>8}")

print("-"*110)
bull_beat = sum(1 for r in bull_results if r['beat'])
bear_beat = sum(1 for r in bear_results if r['beat'])
print(f"Bull Markets: {bull_beat}/{len(bull_results)} beat ({bull_beat/len(bull_results)*100:.0f}%)")
print(f"Bear Markets: {bear_beat}/{len(bear_results)} beat ({bear_beat/len(bear_results)*100:.0f}%)")

---
## 7. Summary

In [ ]:
print("\n" + "="*70)
print("WALK-FORWARD VALIDATION SUMMARY")
print("="*70)

print(f"\n📊 FULL PERIOD (2019-2026):")
print(f"   Return: {pf.total_return()*100:+,.0f}%")
print(f"   Sharpe: {pf.sharpe_ratio():.2f}")
print(f"   Max DD: {pf.max_drawdown()*100:.1f}%")

print(f"\n📅 YEAR-BY-YEAR:")
print(f"   Beat Rate: {beat_count}/{len(yearly_results)} years ({beat_count/len(yearly_results)*100:.0f}%)")

print(f"\n🔄 WALK-FORWARD:")
print(f"   Beat Rate: {wf_beat}/{len(wf_results)} windows ({wf_beat/len(wf_results)*100:.0f}%)")

print(f"\n📈 BULL MARKETS: {bull_beat}/{len(bull_results)} beat ({bull_beat/len(bull_results)*100:.0f}%)")
print(f"📉 BEAR MARKETS: {bear_beat}/{len(bear_results)} beat ({bear_beat/len(bear_results)*100:.0f}%)")

overall_beat = (beat_count/len(yearly_results) + wf_beat/len(wf_results)) / 2 * 100
print(f"\n🎯 OVERALL CONFIDENCE: {overall_beat:.0f}%")

In [ ]:
# Plot full period
pf.plot().show()